In [8]:
# imports
import os
import cv2
import torch
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from tqdm import tqdm

from sklearn.metrics.pairwise import cosine_similarity

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from retinaface import RetinaFace

from insightface.app import FaceAnalysis

In [ ]:
# paths
import os

ROOT_DIR = "naturally_degraded_processes"

GALLERY_DIR = os.path.join(
    ROOT_DIR,
    "gallery",
)

PROBE_DIR = os.path.join(
    ROOT_DIR,
    "probe",
)

CROPPED_GALLERY_DIR = os.path.join(
    ROOT_DIR,
    "gallery_cropped",
)

GALLERY_EMBED_DIR = os.path.join(
    ROOT_DIR,
    "gallery_embeddings",
)

PROBE_EMBED_DIR = os.path.join(
    ROOT_DIR,
    "probe_embeddings",
)

for path in [
    CROPPED_GALLERY_DIR,
    GALLERY_EMBED_DIR,
    PROBE_EMBED_DIR,
]:
    os.makedirs(path, exist_ok=True)

In [11]:
import cv2
import time

identity = os.listdir(GALLERY_DIR)[0]

person_dir = os.path.join(
    GALLERY_DIR,
    identity,
)

img_name = os.listdir(person_dir)[0]

img_path = os.path.join(
    person_dir,
    img_name,
)

print("Image Path:", img_path)

img = cv2.imread(img_path)

print("Image Loaded")

print("Shape:", img.shape)

start = time.time()

faces = RetinaFace.detect_faces(img)

print("Detection Done")

print("Time:", time.time() - start)

print(type(faces))

Image Path: naturally_degraded_processes/gallery/Zendaya/Zendaya_001.jpg
Image Loaded
Shape: (554, 489, 3)
Detection Done
Time: 2.4174458980560303
<class 'dict'>


In [13]:
# cropping the gallery images

saved_count = 0
cropped_count = 0
fallback_count = 0
failed_count = 0

for i, identity in enumerate(tqdm(os.listdir(GALLERY_DIR))):

    person_dir = os.path.join(
        GALLERY_DIR,
        identity,
    )

    if not os.path.isdir(person_dir):
        continue

    files = os.listdir(person_dir)

    if len(files) == 0:
        print("Empty folder:", identity)
        failed_count += 1
        continue

    img_name = files[0]

    img_path = os.path.join(
        person_dir,
        img_name,
    )

    img = cv2.imread(img_path)

    if img is None:
        print("Could not read:", identity)
        failed_count += 1
        continue

    save_dir = os.path.join(
        CROPPED_GALLERY_DIR,
        identity,
    )

    os.makedirs(
        save_dir,
        exist_ok=True,
    )

    save_path = os.path.join(
        save_dir,
        img_name,
    )

    try:

        faces = RetinaFace.detect_faces(img)

        if isinstance(faces, dict) and len(faces) > 0:

            best_face = max(
                faces.values(),
                key=lambda x: (x["facial_area"][2] - x["facial_area"][0])
                * (x["facial_area"][3] - x["facial_area"][1]),
            )

            x1, y1, x2, y2 = best_face["facial_area"]

            h, w = img.shape[:2]

            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(w, x2)
            y2 = min(h, y2)

            crop = img[y1:y2, x1:x2]

            if crop.size > 0:

                cv2.imwrite(
                    save_path,
                    crop,
                )

                cropped_count += 1
                saved_count += 1

            else:

                cv2.imwrite(
                    save_path,
                    img,
                )

                fallback_count += 1
                saved_count += 1

        else:

            # no face detected
            cv2.imwrite(
                save_path,
                img,
            )

            fallback_count += 1
            saved_count += 1

    except Exception as e:

        print(f"{identity} -> {e}")

        # save original image as fallback
        cv2.imwrite(
            save_path,
            img,
        )

        fallback_count += 1
        saved_count += 1

    if (i + 1) % 10 == 0:

        print(
            f"Processed {i+1} identities | "
            f"Saved={saved_count} | "
            f"Cropped={cropped_count} | "
            f"Fallback={fallback_count}"
        )

print("\n========== FINAL SUMMARY ==========")
print("Total Saved      :", saved_count)
print("Successfully Cropped :", cropped_count)
print("Fallback Originals   :", fallback_count)
print("Failed               :", failed_count)

 11%|█         | 10/95 [00:22<02:53,  2.04s/it]

Processed 10 identities | Saved=10 | Cropped=6 | Fallback=4


 21%|██        | 20/95 [00:42<02:33,  2.05s/it]

Processed 20 identities | Saved=20 | Cropped=11 | Fallback=9


 32%|███▏      | 30/95 [01:03<02:08,  1.98s/it]

...

Processed 80 identities | Saved=80 | Cropped=38 | Fallback=42


 95%|█████████▍| 90/95 [03:15<00:09,  1.95s/it]

Processed 90 identities | Saved=90 | Cropped=43 | Fallback=47


100%|██████████| 95/95 [03:25<00:00,  2.16s/it]


========== FINAL SUMMARY ==========
Total Saved      : 95
Successfully Cropped : 46
Fallback Originals   : 49
Failed               : 0


In [14]:
# loading arcface
print("Loading ArcFace...")

app = FaceAnalysis(name="buffalo_l")

app.prepare(
    ctx_id=0,
    det_size=(640, 640),
)

rec_model = app.models["recognition"]

print("ArcFace Ready")

Loading ArcFace...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 

In [15]:
# generating gallery embeddings
gallery_embeddings = {}
for identity in tqdm(os.listdir(CROPPED_GALLERY_DIR)):

    person_dir = os.path.join(
        CROPPED_GALLERY_DIR,
        identity,
    )

    img_name = os.listdir(person_dir)[0]

    img_path = os.path.join(
        person_dir,
        img_name,
    )

    img = cv2.imread(img_path)

    img = cv2.resize(
        img,
        (112, 112),
    )

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB,
    )

    emb = rec_model.get_feat(img).flatten()

    gallery_embeddings[identity] = emb.astype(np.float32)

100%|██████████| 95/95 [00:09<00:00,  9.72it/s]


In [17]:
# probe embeddings

probe_embeddings = {}

for identity in tqdm(os.listdir(PROBE_DIR)):

    person_dir = os.path.join(
        PROBE_DIR,
        identity,
    )

    # skip files like .DS_Store
    if not os.path.isdir(person_dir):
        continue

    files = [
        f for f in os.listdir(person_dir)
        if not f.startswith(".")
    ]

    if len(files) == 0:
        continue

    img_name = files[0]

    img_path = os.path.join(
        person_dir,
        img_name,
    )

    img = cv2.imread(img_path)

    if img is None:
        print("Could not read:", img_path)
        continue

    img = cv2.resize(
        img,
        (112, 112),
    )

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB,
    )

    emb = rec_model.get_feat(img).flatten()

    probe_embeddings[identity] = emb.astype(np.float32)

print("Probe embeddings generated:", len(probe_embeddings))

100%|██████████| 96/96 [00:10<00:00,  9.49it/s]

Probe embeddings generated: 95


In [18]:
# quality score using MUSIQ
quality_scores = {}
for identity in tqdm(os.listdir(PROBE_DIR)):

    person_dir = os.path.join(
        PROBE_DIR,
        identity,
    )

    img_name = os.listdir(person_dir)[0]

    img_path = os.path.join(
        person_dir,
        img_name,
    )

    score = get_quality_score(img_path)

    quality_scores[identity] = score

  0%|          | 0/96 [00:00<?, ?it/s]


NameError: name 'get_quality_score' is not defined

In [ ]:
rows = []

for probe_id, probe_emb in tqdm(probe_embeddings.items()):

    identity_scores = {}

    for gallery_id, gallery_emb in gallery_embeddings.items():

        sim = cosine_similarity(probe_emb.reshape(1, -1), gallery_emb.reshape(1, -1))[
            0
        ][0]

        identity_scores[gallery_id] = sim

    sorted_scores = sorted(identity_scores.items(), key=lambda x: x[1], reverse=True)

    predicted_identity = sorted_scores[0][0]

    best_similarity = sorted_scores[0][1]

    second_similarity = sorted_scores[1][1]

    margin = best_similarity - second_similarity

    label = int(predicted_identity == probe_id)

    rows.append(
        {
            "quality_score": quality_scores[probe_id],
            "best_similarity": best_similarity,
            "margin": margin,
            "label": label,
            "true_identity": probe_id,
            "predicted_identity": predicted_identity,
        }
    )

In [ ]:
df = pd.DataFrame(rows)

df.to_csv(
    "tiny_face_like_features.csv",
    index=False,
)
print(df["label"].value_counts())
df.head()

In [ ]:
# loading the model
class MLPExperiment2(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )

    def forward(self, x):

        return self.network(x)

In [ ]:
model = MLPExperiment2()

model.load_state_dict(
    torch.load("mlp_exp2_model.pth", map_location=torch.device("cpu"))
)

model.eval()

print("MLP Loaded")

In [ ]:
# loading the scaler
scaler = joblib.load("mlp_scaler_exp2.pkl")

print("Scaler Loaded")

In [ ]:
# evaluation function
def evaluate_dataset(csv_file):

    df = pd.read_csv(csv_file)

    X = df[["quality_score", "best_similarity", "margin"]]

    y_true = df["label"]

    X_scaled = scaler.transform(X)

    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

    with torch.no_grad():

        outputs = model(X_tensor)

        y_pred = torch.argmax(outputs, dim=1).numpy()

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(y_true, y_pred, zero_division=0)

    recall = recall_score(y_true, y_pred, zero_division=0)

    f1 = f1_score(y_true, y_pred, zero_division=0)

    cm = confusion_matrix(y_true, y_pred)

    if cm.shape == (2, 2):

        TN, FP, FN, TP = cm.ravel()

        TAR = TP / (TP + FN) if (TP + FN) > 0 else np.nan

        FRR = FN / (TP + FN) if (TP + FN) > 0 else np.nan

        TRR = TN / (TN + FP) if (TN + FP) > 0 else np.nan

        FAR = FP / (TN + FP) if (TN + FP) > 0 else np.nan

    else:

        TAR = np.nan
        FRR = np.nan
        TRR = np.nan
        FAR = np.nan

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TAR": TAR,
        "FRR": FRR,
        "TRR": TRR,
        "FAR": FAR,
        "y_true": y_true,
        "y_pred": y_pred,
    }

In [ ]:
# threshold evaluation
def evaluate_threshold(csv_file):

    df = pd.read_csv(csv_file)

    y_true = df["label"]

    threshold_pred = (df["best_similarity"] >= 0.6).astype(int)

    accuracy = accuracy_score(y_true, threshold_pred)

    precision = precision_score(y_true, threshold_pred, zero_division=0)

    recall = recall_score(y_true, threshold_pred, zero_division=0)

    f1 = f1_score(y_true, threshold_pred, zero_division=0)

    cm = confusion_matrix(y_true, threshold_pred)

    if cm.shape == (2, 2):

        TN, FP, FN, TP = cm.ravel()

        TAR = TP / (TP + FN) if (TP + FN) > 0 else np.nan

        FRR = FN / (TP + FN) if (TP + FN) > 0 else np.nan

        TRR = TN / (TN + FP) if (TN + FP) > 0 else np.nan

        FAR = FP / (TN + FP) if (TN + FP) > 0 else np.nan

    else:

        TAR = np.nan
        FRR = np.nan
        TRR = np.nan
        FAR = np.nan

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TAR": TAR,
        "FRR": FRR,
        "TRR": TRR,
        "FAR": FAR,
        "y_true": y_true,
        "y_pred": threshold_pred,
    }

In [ ]:
# running our multi layer MLP
mlp_exp2_results = {}

for dataset_name, csv_file in datasets.items():

    result = evaluate_dataset(csv_file)

    mlp_exp2_results[dataset_name] = result

    print("\n======================")
    print(dataset_name)
    print("======================")

    print(pd.DataFrame([result]).round(4))

In [ ]:
# running fixed threshold method
threshold_results = {}

for dataset_name, csv_file in datasets.items():

    result = evaluate_threshold(csv_file)

    threshold_results[dataset_name] = result

    print("\n======================")
    print(dataset_name)
    print("======================")

    print(pd.DataFrame([result]).round(4))

In [ ]:
# final comparison
comparison = pd.DataFrame(
    {
        "Method": [
            "Threshold",
            "MLP Exp2",
        ],
        "Accuracy": [
            threshold_accuracy,
            mlp_accuracy,
        ],
        "Precision": [
            threshold_precision,
            mlp_precision,
        ],
        "Recall": [
            threshold_recall,
            mlp_recall,
        ],
        "F1": [
            threshold_f1,
            mlp_f1,
        ],
        "TAR": [
            threshold_tar,
            mlp_tar,
        ],
        "FRR": [
            threshold_frr,
            mlp_frr,
        ],
        "TRR": [
            threshold_trr,
            mlp_trr,
        ],
        "FAR": [
            threshold_far,
            mlp_far,
        ],
    }
)

comparison.round(4)